當今天有一份大檔案，使用者要 Claude 根據這份檔案回答他的問題時，有以下兩種方法可以讓 Claude 進行

1. Include Everything in the Prompt : 我們可以像以下方式將檔案內容提供給 Claude

    ```json
    Answer the user's question about the financial document.

    <user_question>
    {user_question}
    </user_question>

    <financial_document>
    {financial_document}
    </financial_document>
    ```
但會有以下限制:
- There's a **hard limit on prompt length** - your document might be too long
- Claude becomes **less effective with very long prompts**
- Larger prompts **cost more** to process
- Larger prompts **take longer** to process

![](./figure/04/RAG_intro_01.jpg)
![](./figure/04/RAG_intro_02.jpg)

---

2. Break Documents into Chunks (RAG)
透過前處理將檔案分割成較小的片段 (`chunk`)，後續要使用時尋找最相關的 chunk 提供給 Claude

使用 RAG 有以下特性:
- 優點:
    - Claude can **focus on only the most relevant content**
    - Scales up to very large documents
    - Works with multiple documents
    - Smaller prompts **cost less and run faster**

- 困難點:
    - Requires a **preprocessing step** to chunk documents
    - Need a search mechanism to **find "relevant" chunks**
    - Included chunks **might not contain all the context Claude needs**
    - Many ways to chunk text - which approach is best?

![](./figure/04/RAG_intro_03.jpg)
![](./figure/04/RAG_intro_04.jpg)

---

When to Use RAG?

> It's especially valuable when working with very large documents, multiple documents, or when you need to optimize for cost and performance. 

> *The key insight is that RAG trades simplicity for scalability and efficiency.*

# Text chunking strategies

範例可看 : [./source/001_chunking.ipynb](./source/001_chunking.ipynb)

![](./figure/04/chunk_strategy_01.jpg)

### Size-Based Chunking

將檔案均分成固定 *字數* 長度的 `chunk`

- 優點:
    - 最簡單
- 缺點:
    - 字可能在句子中間被截斷
    - Chunks lose important context from surrounding text
    - Section headers might be separated from their content

![](./figure/04/chunk_strategy_02.jpg)

一個簡易的解決方法，讓兩個相鄰的 chunk 之間重複 (overlap) 一些內容，能夠提高完整的句子或字詞的機率

```python
def chunk_by_char(text, chunk_size=150, chunk_overlap=20):
    chunks = []
    start_idx = 0
    
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunk_text = text[start_idx:end_idx]
        chunks.append(chunk_text)
        
        start_idx = (
            end_idx - chunk_overlap if end_idx < len(text) else len(text)
        )
    
    return chunks
```

![](./figure/04/chunk_strategy_03.jpg)
![](./figure/04/chunk_strategy_04.jpg)

### Structure-Based Chunking

根據檔案內的結構進行 chunk 分割

- 優點:
    - 當檔案結構良好時，透過此方法可以獲得高品質的 chunk
- 缺點:
    - 只在檔案結構良好時可以使用，像是 plaintext 或是 pdf 等檔案都不適用

```python
def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)
```

![](./figure/04/chunk_strategy_05.jpg)

### Semantic-Based Chunking

將內容以句子單位分割，並透過自然語言處理將相關的句子 group 起來

- 優點:
    - 能夠將最相關的句子整理在一個 chunk
- 缺點:
    - 最複雜
    - 需要最多計算

### Sentence-Based Chunking

將檔案內容以正則表達式分割成獨立的句子，並用可調整的 overlap 進行 group，是最實際可行的折衷方案

```python
def chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    
    chunks = []
    start_idx = 0
    
    while start_idx < len(sentences):
        end_idx = min(start_idx + max_sentences_per_chunk, len(sentences))
        current_chunk = sentences[start_idx:end_idx]
        chunks.append(" ".join(current_chunk))
        
        start_idx += max_sentences_per_chunk - overlap_sentences
        
        if start_idx < 0:
            start_idx = 0
    
    return chunks
```

### Choosing Your Strategy

Your choice depends entirely on your **use case** and **document guarantees**:

- **Structure-based** : Best results when you control document formatting (like internal company reports)
- **Sentence-based** : Good middle ground for most text documents
- **Size-based** : Most reliable fallback that works with any content type, including code

Size-based chunking with overlap is often the go-to choice in production because it's simple, reliable, and works with any document type. While it may not give perfect results, it consistently produces reasonable chunks that won't break your pipeline.

**Remember**: **there's no single "best" chunking strategy.** The right approach depends on your specific documents, use cases, and the trade-offs you're willing to make between implementation complexity and chunk quality.

# Text embeddings

尋找與使用者提問最相關的 chunk

![]()

## Semantic Search

`semantic search` uses **text embeddings** to understand the meaning and context of both the user's question and each text chunk

### Text Embeddings

A text embedding is a numerical representation of the meaning contained in some text. Think of it as converting words and sentences into a format that computers can work with mathematically.

Here's how the process works:

- You feed text into an embedding model
- The model outputs a long list of numbers (the embedding)
- Each number ranges `from -1 to +1`
- These numbers represent different qualities or features of the input text

### `VoyageAI` for Embeddings

Anthropic 目前沒有推出 embedding generation，推薦 `VoyageAI`，使用前需要:

1. Sign up for a separate VoyageAI account
2. Get an API key (free to get started)
3. Add the key to your environment variables

(以上可以查看 [./source/VoyageAI_API_Key_Directions.pdf](./source/VoyageAI_API_Key_Directions.pdf))

在 `.env` 檔案裡新增:

```env
VOYAGE_API_KEY="your_key_here"
```

### Implementation

> 範例在 [./source/002_embeddings.ipynb](./source/002_embeddings.ipynb)

# Full RAG Flow


## STEP 1. Chunk Your Source Text

## STEP 2. Generate Embeddings

### 1. 將分割好的 chunk 丟進 `embedding model` 進行向量化

![](./figure/04/RAG_flow_01.jpg)

### 2. 將向量化完成之 embedding 進行 **`Normalization`**

![](./figure/04/RAG_flow_02.jpg)
![](./figure/04/RAG_flow_03.jpg)

## STEP 3. 將第二步產生之所有 Embeddings Store in Vector Database

這步驟完成後我們的前置處理就結束了，等待 user request

![](./figure/04/RAG_flow_04.jpg)

## STEP 4. Process User Query
收到 user 發送之 request，將其放入一開始我們使用的相同 embedding model 產生 request 的 embedding

![](./figure/04/RAG_flow_05.jpg)

## STEP 5. Find Similar Embeddings

### 1. 計算此 request 與每一個 Vector Database 裡的 embedding 之間的 Cosine Similarity

![](./figure/04/RAG_flow_06.jpg)

### 2. 根據 Cosine Similarity 計算 Cosine Distance : `(1 - cosine similarity)`
- Values close to 0 mean high similarity
- Larger values mean less similarity

![](./figure/04/RAG_flow_07.jpg)

## STEP 6. Create the Final Prompt

將 user request 與以上步驟計算出來最相關的 chunk 放入 prompt 中，傳送給 Claude 以獲得回覆

Prompt 會長得像這樣:

```xml
Answer the user's question about the financial document.

<user_question>
How many bugs did engineers fix this year?
</user_question>

<report>
## Section 2: Software Engineering
This division dedicated significant effort to studying various infection vectors in our distributed systems
</report>
```

![](./figure/04/RAG_flow_08.jpg)

## Implementation

> 詳見: [./source/003_vectordb.ipynb](./source/003_vectordb.ipynb)

# BM25 lexical search

## The Problem with Semantic Search Alone

語意搜尋有可能會回傳*相關*但並不是你想要找的主要內容，這時候我們可以導入另一個系統來進行平衡

> Semantic search focuses on conceptual similarity rather than exact term matching

## Hybrid Search Strategy

可以加入 **字詞搜尋(`Lexical searches`)** 系統

- **Semantic search** finds conceptually related content using embeddings
- **Lexical search** finds exact term matches using classic text search
- **Merged results** combine both approaches for better accuracy

![](./figure/04/BM25_01.jpg)

## Recommend Lexical searches : **BM25**

BM25 工作流程:
1. **Tokenize the query** : Break the user's question into individual terms. 
    
    For example, "a INC-2023-Q4-011" becomes ["a", "INC-2023-Q4-011"].

2. **Count term frequency** : See how often each term appears across all your documents. Common words like "a" might appear 5 times, while specific terms like "INC-2023-Q4-011" might appear only once.

3. **Weight terms by importance** : Terms that appear less frequently get higher importance scores. The word "a" gets low importance because it's common, while "INC-2023-Q4-011" gets high importance because it's rare.

4. **Find best matches** : Return documents that contain more instances of the higher-weighted terms.

![](./figure/04/BM25_02.jpg)

## Implementation

> 詳見 [./source/004_bm25.ipynb](./source/004_bm25.ipynb)

## Why This Works Better

BM25 excels at finding exact matches because it:

- Gives higher weight to rare, specific terms
- Ignores common words that don't add search value
- Focuses on term frequency rather than semantic meaning
- Works especially well for technical terms, IDs, and specific phrases

The key insight is that both search methods have complementary strengths. **Semantic search** *understands context and meaning*, while **lexical search** ensures you *don't miss exact term matches*. 

By combining them, you create a **more robust** search system that *handles both conceptual queries and specific lookups* effectively.

# A Multi-Index RAG pipeline

## The Multi-Index Architecture

透過新增一個 `Retriever` class 把之前我們寫好的兩個 search system combine 起來，也是因為要串起來才把他們兩個 search API (`add_document()`、`search()` method) 寫的一模一樣

-> 之後如果要擴充其他的 search system，只要寫一樣的 API 就可以整合，有良好的擴容性

![](./figure/04/Multi-Index_Rag_Pipeline_01.jpg)

## Understanding `Reciprocal Rank Fusion (RRF)`

兩個系統擁有兩個不同評分標準，因此我們需要一個方法公平的進行整合

![](./figure/04/Multi-Index_Rag_Pipeline_02.jpg)
![](./figure/04/Multi-Index_Rag_Pipeline_03.jpg)

**The RRF formula:** `RRF_score(d) = Σ(1 / (k + rank_i(d)))`

![](./figure/04/Multi-Index_Rag_Pipeline_04.jpg)

## Implementation Details

> 詳見 [./source/005_hybrid.ipynb](./source/005_hybrid.ipynb) 